## **Step 1: At the start of every session**

#### Cell A: Drive, wandb, and the GPU

In [1]:
from google.colab import drive, userdata
import json, os, pathlib, subprocess
drive.mount("/content/drive")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
DRIVE = pathlib.Path("/content/drive/MyDrive/msml612")
DATASET = DRIVE / "physionet_set_a.npz"
!nvidia-smi --query-gpu=name --format=csv,noheader

Mounted at /content/drive
NVIDIA A100-SXM4-80GB


This cell must print NVIDIA A100-SXM4-80GB. If it prints anything else, disconnect and try again later. Never use another GPU.

#### Cell B: the code, pinned to the sweep's commit

In [2]:
!git clone -q https://github.com/jkelle11-source/msml612_project.git
%cd /content/msml612_project
started = DRIVE / "sweep" / "sweep.json"
if started.exists():        # a later session: the commit the sweep started from
    subprocess.run(["git", "checkout", "-q", json.loads(started.read_text())["commit"]], check=True)
!git log --oneline -1
!uv sync --locked
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!sha256sum {DATASET}

/content/msml612_project
20a470d (HEAD, origin/main, origin/HEAD, main) Merge pull request #58 from jkelle11-source/m6-2-results_sankha
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 67 packages in 39ms
Prepared 63 packages in 24.20s
Installed 63 packages in 327ms
 + annotated-types==0.8.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + einops==0.8.2
 + filelock==4.0.12
 + fsspec==2026.9.0
 + googleapis-common-protos==1.75.5
 + idna==3.20
 + iniconfig==2.3.1
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.5.3
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.24.0.43
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 +

Check three things:
- **The commit:** the first session should show `20a470d Merge pull request #58`, or the newest merge if something else lands on `main` first.
- **The torch line:** `2.14.1+cu130 13.0 True`.
- **The dataset hash:** `ce6ec696…7458070`.

## **Step 2: The sweep, 30 cells, about 61 h over several sessions**

#### Cell C

In [ ]:
!uv run python -m diffusion.sweep --tag sweep --out {DRIVE}/sweep --dataset {DATASET}

30 cells on cuda -> /content/drive/MyDrive/msml612/sweep
[1/30] base_random_seed42
  trained
  scored on the random holdout
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
  scored on the block holdout
[2/30] ext1_random_seed42
  trained
  scored on the random holdout
  scored on the block holdout
[3/30] ext2_random_seed42
  trained
  scored on the random holdout
  scored on the block holdout
[4/30] ext1+2_random_seed42
  trained
  scored on the random holdout
  scored on the block holdout
[5/30] base_random_seed43
  trained
  scored on the random holdout
  scored on the block holdout
[6/30] ext1_random_seed43
  trained
  scored on the random holdout
  scored on the block holdout
[7/30] ext2_random_seed43
  trained
  scored on the random holdout
  scored on the block holdout
[8/30] ext1+2_random_seed43
  trained
  scored on the random holdout
  scored on the block holdout
[9/30] base_random_seed44
  trained
  scored on the random holdout
  scored on

- **The first lines:** `30 cells on cuda -> /content/drive/MyDrive/msml612/sweep`, then `[1/30] base_random_seed42`.
- **The order:** the random regime first (seeds 42-46, four variants each), then block-aware (seeds 42-46, base and ext1).
- **Each cell takes about 2 h:** 25-36 min of training, about 14 min of validation, then two scoring passes of 39 min each.
- **Early check:** about 40 minutes in, wandb should show `base_random_seed42` tagged `sweep`.
- **If a session is lost:** start a new one and run A, B, and C again. Finished cells print `trained` and `scored on the ... holdout`. An unfinished cell either resumes from its checkpoint or starts again.
- **Ending a session yourself:** run `drive.flush_and_unmount()` first.
- **Done** when it prints `done: collect and check them with ... diffusion.results --tag sweep`.

## Step 3: Base's 50% and 90% scorings, 10 passes, about 6.5 h

Run this only after Cell C prints `done:`. In a new session, run A and B, then this cell; Cell C isn't needed again. It skips any pass already done and adds it to wandb again, in case a session ended before it got there.

#### Cell D

In [ ]:
%%bash
cd /content/msml612_project
SWEEP=/content/drive/MyDrive/msml612/sweep
DATASET=/content/drive/MyDrive/msml612/physionet_set_a.npz
for seed in 42 43 44 45 46; do
  for f in 0.5 0.9; do
    run=$SWEEP/base_random_seed$seed
    scores=$run/scores_test_random_$f.json
    if [ -f "$scores" ] && [ -f "$run/median_test_random_$f.npy" ]; then
      echo "$(basename $run) at $f: already scored, adding it to wandb again"
      uv run python -c "import json, sys; from diffusion.score import attach_to_wandb; print(attach_to_wandb(json.load(open(sys.argv[1]))))" "$scores"
    else
      uv run python -m diffusion.score "$run" --fraction $f --paired --dataset "$DATASET"
    fi
  done
done

Each new pass ends with `added to the wandb summary of ...`. If units run short, change `42 43 44 45 46` to `42 43 44`, as the gate did, and note the cut.